# vla.cpp - run SmolVLA on LIBERO (Colab)

Runs the steps of [apps/vla.cpp/README.md](README.md) on a Colab GPU runtime:

1. build the `vla-server` with CUDA,
2. create the LIBERO simulation env for the client,
3. download the SmolVLA GGUF bundle,
4. start the server, run one LIBERO episode, and play the output video.

**Runtime:** `Runtime -> Change runtime type -> GPU`, then run the cells top to bottom.
The build and the LIBERO env each take roughly 10-20 minutes the first time.

Two differences from the README, because upstream `vla.cpp` changed:

- there is no `patches/patch.sh` any more - CMake now patches llama.cpp itself;
- the bundle is a single `smolvla-libero.gguf` (no `mmproj-*.gguf`), so the server takes one file.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## 1. Build the server

`vla.cpp`'s CUDA build needs an **Ampere or newer GPU (`sm_80`+)**: its FoldQuant kernels use
`mma.m16n8k64` and `cp.async`, which older GPUs lack, and CMake always builds them. So:

- **L4 / A100** (Colab Pro): CUDA build, as in the README.
- **T4** (free Colab, `sm_75`): the CUDA build fails in `ptxas`, so this cell builds the
  **CPU** server instead. It works, just slower per action.

In [ ]:
%%bash
set -euxo pipefail
apt-get -qq update
apt-get -qq install -y libzmq3-dev cppzmq-dev libprotobuf-dev protobuf-compiler > /dev/null

cd /content
[ -d vla.cpp ] || git clone --depth 1 https://github.com/VinRobotics/vla.cpp.git
cd vla.cpp

export PATH=/usr/local/cuda/bin:$PATH
CUDA_ARCHITECTURE=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
if [ "$CUDA_ARCHITECTURE" -ge 80 ]; then
    CUDA=ON
else
    CUDA=OFF
    echo "sm_$CUDA_ARCHITECTURE < sm_80: vla.cpp CUDA kernels need Ampere+, building the CPU server"
fi
# a build dir configured for the other backend (e.g. a failed CUDA try) is reconfigured from scratch
grep -qs "GGML_CUDA:BOOL=$CUDA" build/CMakeCache.txt || rm -rf build
cmake -B build \
    -DGGML_CUDA=$CUDA \
    -DGGML_CUDA_GRAPHS=ON \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_CUDA_ARCHITECTURES=$CUDA_ARCHITECTURE
cmake --build build -j$(nproc)
ls -lh build/vla-server

## 2. Create the LIBERO simulation env with uv

`setup_libero.sh` clones LIBERO and builds a Python 3.10 venv in `eval/sim/libero/libero_uv/.venv`.
MuJoCo renders headless through EGL on the GPU; the snippet registers NVIDIA's EGL driver,
which Colab images do not always ship a config for.

Colab presets `UV_SYSTEM_PYTHON` (and other `UV_*` vars), which makes `uv pip install` ignore the
new venv and install into Colab's own Python 3.13 - where LIBERO's old pins (e.g. `pathtools`,
which imports the removed `imp` module) fail to build. The cell clears them first.

In [ ]:
%%bash
set -euxo pipefail
pip -q install uv
for v in $(env | grep -o '^UV_[A-Z_]*' || true); do unset "$v"; done
ICD=/usr/share/glvnd/egl_vendor.d/10_nvidia.json
if [ ! -f "$ICD" ]; then
  mkdir -p "$(dirname "$ICD")"
  echo '{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}' > "$ICD"
fi
cd /content/vla.cpp
bash eval/sim/libero/setup_libero.sh

## 3. Get the SmolVLA bundle

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download("vrfai/smolvla-libero-gguf", local_dir="/content/vla.cpp/models/smolvla")
!ls -lh /content/vla.cpp/models/smolvla

## 4. Run

Start the server in the background and wait until it reports it is bound and ready.
The `vla: backend = ...` line says where it runs (`CUDA` on L4/A100, `CPU` on a T4).

In [ ]:
import re, subprocess, time

REPO = "/content/vla.cpp"
LOG = "/content/vla-server.log"

server = subprocess.Popen(
    ["./build/vla-server", "--bind", "tcp://*:5555", "models/smolvla/smolvla-libero.gguf"],
    cwd=REPO, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
)
for _ in range(600):
    log = open(LOG).read()
    if re.search(r"bound to .* ready", log):
        break
    if server.poll() is not None:
        raise RuntimeError("vla-server exited:\n" + log[-3000:])
    time.sleep(1)
else:
    raise TimeoutError("vla-server not ready after 600s:\n" + log[-3000:])

print(log[-2000:])
print([l for l in log.splitlines() if "backend =" in l])

Run one LIBERO episode (the README's client command, using the venv's Python directly
because `source .venv/bin/activate` does not carry across notebook cells).

`--n-action-steps 10` executes 10 actions of each predicted chunk before asking the server
again - 10x fewer model calls, which matters on the CPU server. It is also what upstream's
LIBERO sweep uses for SmolVLA (the SmolVLA paper found 10 beats 1).

In [ ]:
%%bash
set -euxo pipefail
cd /content/vla.cpp
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl
eval/sim/libero/libero_uv/.venv/bin/python eval/client/run_sim_client_direct.py \
    --task libero_object --task-id 0 --n-episodes 1 \
    --output-dir /tmp/libero_outputs \
    --arch smolvla --n-action-steps 10

## 5. Result

The client writes the video, `summary.txt` and `episodes.csv` to
`/tmp/libero_outputs/smolvla/libero_object/task_0/`.

In [ ]:
import base64, glob
from IPython.display import HTML, display

out = "/tmp/libero_outputs/smolvla/libero_object/task_0"
print(open(f"{out}/summary.txt").read())
video = sorted(glob.glob(f"{out}/*.mp4"))[-1]
b64 = base64.b64encode(open(video, "rb").read()).decode()
display(HTML(f'<video width="640" controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

In [ ]:
server.terminate()   # stop the background vla-server